# GitHub for KSB Project
### UFC / Polymarket — Code and Data Check-In
**Matthew Cohen and Oliver Cherwin**  
**November 15, 2026**

## Research question
I am using my UFC prediction model as the starting point for our project. We want to find out whether a model built on public fighter statistics can identify mispriced UFC contracts on Polymarket.

**We test the hypothesis that** Polymarket UFC prices already reflect publicly available fighter statistics, so a model that only uses information known before each fight cannot consistently earn positive returns after trading costs.

## Economic story
One reason prices might be wrong is fan bias. A popular fighter may get more support because fans know their name or want them to win, even when the price is too high. If our model finds a real opportunity, fans or other traders willing to overpay could be on the other side. We also want to know whether higher trading volume makes markets more accurate or just brings in more fans. This is something we need to test, not assume.

## Data and where we are now
My existing UFC model uses performance measures such as recent results, striking, takedowns, reach, age, and an opponent-adjusted rating. For the class project, we plan to use UFCStats for fight records and statistics and Polymarket for historical prices. We want one observation per fight at a consistent time before the fight starts.

Below is a **12-row example** taken from my existing GitHub prediction log, rather than the final class dataset. The log labels market numbers as recovered pre-fight snapshots, but the saved rows do not contain direct market URLs, so we still have to verify those prices. The code is just an initial data check.

## Method and next steps
For now, we calculate the difference between the model's probability and the saved market probability. That tells us where they disagree, **not** whether a bet would have made money. Next we need a larger dataset, verified prices with timestamps, and an out-of-sample comparison against a simpler model such as logistic regression. We will account for trading costs and report whether any results are statistically convincing.

## Reflection
The hardest part so far is turning a working prediction app into a fair historical test. We need to make sure older fights are predicted only with data available at the time, not the current fighter statistics or a model retrained on later results. We also need to account for fights that do not have usable Polymarket prices.

Source repository: https://github.com/mpc4900/ufc-edge-model


## Starter code
Small sample for testing the data structure, not final results.

In [ ]:
# Load a small saved sample from my existing UFC model
# It is included here so there are no extra files to upload.
from io import StringIO
import pandas as pd

sample_csv = """event_date,fighter_a,fighter_b,pick,model_probability,market_probability
2026-08-15,Islam Makhachev,Ian Machado Garry,Ian Machado Garry,0.396,0.255
2026-08-15,Mackenzie Dern,Gillian Robertson,Gillian Robertson,0.663,0.355
2026-08-15,Jalin Turner,Kaue Fernandes,Jalin Turner,0.58,0.505
2026-08-15,Mansur Abdul-Malik,Dustin Stoltzfus,Dustin Stoltzfus,0.254,0.145
2026-08-15,Edson Barboza,Esteban Ribovics,Edson Barboza,0.396,0.145
2026-08-15,Chidi Njokuani,Joel Alvarez,Chidi Njokuani,0.327,0.255
2026-08-15,Charles Johnson,Eduardo Chapolin,Eduardo Chapolin,0.603,0.49
2026-08-15,Donte Johnson,Eric McConico,Donte Johnson,0.778,0.745
2026-08-15,Vicente Luque,Tresean Gore,Vicente Luque,0.669,0.505
2026-08-15,Rafael Tobias,Lucas Fernando,Rafael Tobias,0.466,0.28
2026-08-15,Neil Magny,Ramiz Brahimaj,Neil Magny,0.532,0.425
2026-08-15,Jeremiah Wells,Myktybek Orolbai,Jeremiah Wells,0.249,0.095"""
fights = pd.read_csv(StringIO(sample_csv))
print("Number of saved records:", len(fights))
display(fights.head())


In [ ]:
# Basic checks before comparing the probabilities
fights["event_date"] = pd.to_datetime(fights["event_date"], errors="coerce")
for col in ["model_probability", "market_probability"]:
    fights[col] = pd.to_numeric(fights[col], errors="coerce")

fights = fights.dropna()
valid = (
    fights["model_probability"].between(0, 1)
    & fights["market_probability"].gt(0)
    & fights["market_probability"].lt(1)
)
fights = fights.loc[valid].copy()
print("Usable records:", len(fights))
print("Unique event dates:", fights["event_date"].nunique())


In [ ]:
# A positive gap means my model assigns a higher win probability
# than the saved market number. This is not a profit calculation.
fights["probability_gap_pp"] = 100 * (
    fights["model_probability"] - fights["market_probability"]
)
display(fights[[
    "pick", "model_probability", "market_probability", "probability_gap_pp"
]].round(3))
print("Mean probability gap (percentage points):",
      round(fights["probability_gap_pp"].mean(), 2))
print("Next: verify historical prices and run a proper out-of-sample test.")
